In [6]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import sys
import shutil
import pandas as pd

sys.path.append("../..")
from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()
from copy import copy

from src import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

/tmp/ipykernel_966333/3539475184.py:28: DeprecationWarning: PlottingFunctions.Plotter is deprecated and will be removed in a future version. Use PlottingBase.PublicationPlotter directly instead.
  plotter = PlottingFunctions.Plotter()


In [20]:
def puncta_selector(folder, saving_folder, percentile_lower=80, percentile_higher=85, ROI_size=12, dye='ATTO488', background_threshold=1, n_puncta=10,
                   ):
    files = H_F.file_search(folder, '.h5', dye)
    image_files = H_F.file_search(folder, '.tif', dye)
    data = pd.read_hdf(files[0])
    if 'background_photons' not in data:
        data['background_photons'] = data['bg_B'] + data['bg_R'] + data['bg_G']
    if 'photons' not in data:
        data['photons'] = data['A_B'] + data['A_R'] + data['A_G']
    data = data[data['photons'] < 20000]
    data = data[data['background_photons'] > background_threshold]
    image = IO.read_tiff(image_files[0])
    SBR = np.array(data['photons']/data['background_photons'])
    lower_threshold, upper_threshold = np.percentile(SBR, (percentile_lower, percentile_higher))
    data = data[(lower_threshold < SBR) & (SBR <= upper_threshold)]
    data = data.sort_values('photons', ascending=False).reset_index()
    for i in np.arange(n_puncta):
        x = data['xc'][i]
        y = data['yc'][i]
        frame = int(data['frame'][i])
        xmin = int(int(x) - ROI_size/2)+1
        xmax = int(int(x) + ROI_size/2)+1
        ymin = int(int(y) - ROI_size/2)+1
        ymax = int(int(y) + ROI_size/2)+1
        image_punctum = image[frame, ymin:ymax, xmin:xmax]
        fig, axs = plotter.one_column_plot()
        axs = plotter.image_plot(axs, data=image_punctum, pixelsize=69, scalebarsize=300, scalebarlabel='300 nm', cbar='off')
        plt.savefig(os.path.join(saving_folder, dye+'_punctum_'+str(i).zfill(4)+'.svg'), dpi=600, format='svg')
        plt.close()
    return

In [21]:
saving_folder = '/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Grants/202508_EPSRC_Multicolour/Figures/'
puncta_selector(folder, saving_folder=saving_folder, dye='ATTO647N', percentile_higher=85, percentile_lower=60)